In [1]:
import os
import cv2
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
import torch

class BrainTumorDataset(Dataset):

    def __init__(self, segmentation_root_dir, classification_root_dir, img_size=256):

        self.img_size = img_size
        self.transform = None
        self.class_to_idx = {"glioma": 0, "meningioma": 1, "pituitary": 2, "notumor": 3}
        self.samples =[]

        seg_folder_to_class = {
            "Glioma": "glioma",
            "Meningioma": "meningioma",
            "Pituitary tumor": "pituitary",
        }

        for folder_name, class_name in seg_folder_to_class.items():
            folder_path = os.path.join(segmentation_root_dir, folder_name)
            class_idx = self.class_to_idx[class_name]

            for fname in os.listdir(folder_path):

                if "_mask.png" in fname:
                    continue

                if not fname.endswith((".png", ".jpg", ".jpeg")):
                    continue

                img_path = os.path.join(folder_path, fname)
                stem = os.path.splitext(fname)[0]
                mask_name = f"{stem}_mask.png"
                mask_path = os.path.join(folder_path, mask_name)

                if not os.path.exists(mask_path):
                    continue

                self.samples.append((img_path, mask_path, class_idx))

        for split_folder in ["Training", "Testing"]:
            no_tumor_path = os.path.join(classification_root_dir, split_folder, "notumor")
            notumor_idx = self.class_to_idx["notumor"]
            for fname in os.listdir(no_tumor_path):

                if not fname.endswith((".png", ".jpg", ".jpeg")):
                        continue

                img_path = os.path.join(no_tumor_path, fname)

                self.samples.append((img_path, None, notumor_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path, class_idx = self.samples[idx]

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        # img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if mask_path is not None:
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            mask = (mask > 127).astype(np.float32)

        else:
            h, w = img.shape[:2]
            mask = np.zeros((h, w), dtype=np.float32)

        if self.transform is not None:
            augmented = self.transform(image=img, mask=mask)
            img = augmented["image"]
            mask = augmented["mask"]

        else:
            img = torch.from_numpy(img).unsqueeze(0).float() / 255.0  # (H,W) -> (1,H,W)
            mask = torch.from_numpy(mask)

        if mask.ndim == 2:
            mask = mask.unsqueeze(0)

        class_label = torch.tensor(class_idx, dtype=torch.long)

        return img, mask, class_label

    def load_labels_only(self):
        """Returns just the class index for every sample, in order. Used for
        stratified splitting — sklearn needs the label array without loading
        any actual images."""
        return [c for _, _, c in self.samples]


In [2]:
train_transform = A.Compose([
    A.Resize(256, 256),                  # uniform size — required for batching
    A.HorizontalFlip(p=0.5),             # anatomically plausible for axial MRI slices
    A.Rotate(limit=15, p=0.5),           # small rotation — real scan angle varies slightly
    A.RandomBrightnessContrast(p=0.3),   # simulates scanner/exposure variation
    A.Normalize(mean=(0.5,), std=(0.5,)),  # scale to roughly [-1, 1]
    ToTensorV2(),                        # converts numpy HWC -> torch CHW tensor
])

val_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2(),
])

In [3]:
from collections import Counter

master = BrainTumorDataset(
    segmentation_root_dir="tumor_dataset/DATASET/Segmentation",
    classification_root_dir="tumor_dataset/DATASET/classification",
)

labels = master.load_labels_only()

print(Counter(labels))  # sanity check — confirm class balance matches what we expect

Counter({3: 2000, 2: 930, 1: 708, 0: 554})


In [4]:
labels[:5]

[0, 0, 0, 0, 0]

In [5]:
from sklearn.model_selection import train_test_split

train_idx, temp_idx = train_test_split(
    range(len(master)),
    test_size=0.3,
    stratify=labels,
    random_state=42
)

temp_labels = [labels[i] for i in temp_idx]

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=temp_labels,
    random_state=42
)

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

Train: 2934, Val: 629, Test: 629


In [6]:
class SplitWrapper(Dataset):
    def __init__(self, master_dataset, indices, transform):
        self.master_dataset = master_dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        self.master_dataset.transform = self.transform
        return self.master_dataset[real_idx]

In [7]:
train_dataset = SplitWrapper(master, train_idx, train_transform)
val_dataset = SplitWrapper(master, val_idx, val_transform)
test_dataset = SplitWrapper(master, test_idx, val_transform)

In [8]:
train_loader = DataLoader(train_dataset,
                          batch_size=16,
                          shuffle=True,
                          pin_memory=True,
                          drop_last=True,
                          # num_workers=4
                          )
val_loader   = DataLoader(val_dataset,
                          batch_size=16,
                          shuffle=False,
                          pin_memory=True,
                          # num_workers=4
                          )
test_loader  = DataLoader(test_dataset,
                          batch_size=16,
                          shuffle=False,
                          pin_memory=True,
                          # num_workers=4
                          )

In [9]:
batch = next(iter(train_loader))
images, masks, labels = batch
print("images:", images.shape, images.dtype)
print("masks:", masks.shape, masks.dtype)
print("labels:", labels.shape, labels.dtype)

images: torch.Size([16, 1, 256, 256]) torch.float32
masks: torch.Size([16, 1, 256, 256]) torch.float32
labels: torch.Size([16]) torch.int64


In [10]:
from torch import nn
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class UNet(nn.Module):
    def __init__(self, in_channels, classifier_hidden = 256, num_classes=4, features = [64, 128, 256, 512]):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool2d(2)
        self.final_conv = nn.Conv2d(features[0], 1, kernel_size = 1)
        self.bottleneck = self._double_conv(features[-1], features[-1]*2)
        self.classifier_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(features[-1]*2, classifier_hidden),
            nn.ReLU(inplace=True),
            nn.Linear(classifier_hidden, num_classes)
        )

        for feature in features:
            self.downs.append(self._double_conv(in_channels, feature))
            in_channels = feature

        for feature in reversed(features):
            self.ups.append(nn.ConvTranspose2d(feature*2, feature, kernel_size=2, stride=2))
            self.ups.append(self._double_conv(feature*2, feature))

    def _double_conv(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size = 3, stride = 1, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size = 3, stride = 1, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        # print("Input:", x.shape)

        skip_connections = []

        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)

        pooled_output = self.classifier_pool(x)
        flattened_output = torch.flatten(pooled_output, start_dim=1)
        logits = self.classifier(flattened_output)

        # print("After bottleneck:", x.shape)

        skip_connections = skip_connections[::-1] # reversing the list cause we will be needing the last skip connection first

        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)
            skip_connection = skip_connections[idx//2]
            x = torch.cat([skip_connection, x], dim=1)
            x = self.ups[idx + 1](x)

        mask_pred = self.final_conv(x)

        return mask_pred, logits

In [11]:
x = torch.randn(1, 1, 256, 256)

model = UNet(in_channels=1)

model.eval()

mask, logits = model(x)

print(mask.shape)
print(logits.shape)

torch.Size([1, 1, 256, 256])
torch.Size([1, 4])


In [12]:
from torchinfo import summary

summary(
    model,
    input_size=(1,1,256,256),
    col_names=[
        "input_size",
        "output_size",
        "num_params"
    ]
)

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
UNet                                     [1, 1, 256, 256]          [1, 1, 256, 256]          --
├─ModuleList: 1-7                        --                        --                        (recursive)
│    └─Sequential: 2-1                   [1, 1, 256, 256]          [1, 64, 256, 256]         --
│    │    └─Conv2d: 3-1                  [1, 1, 256, 256]          [1, 64, 256, 256]         640
│    │    └─BatchNorm2d: 3-2             [1, 64, 256, 256]         [1, 64, 256, 256]         128
│    │    └─ReLU: 3-3                    [1, 64, 256, 256]         [1, 64, 256, 256]         --
│    │    └─Conv2d: 3-4                  [1, 64, 256, 256]         [1, 64, 256, 256]         36,928
│    │    └─BatchNorm2d: 3-5             [1, 64, 256, 256]         [1, 64, 256, 256]         128
│    │    └─ReLU: 3-6                    [1, 64, 256, 256]         [1, 64, 256, 256]         --
├─MaxPool2d: 1-2   

In [13]:
import torch.nn as nn

segmentation_criterion = nn.BCEWithLogitsLoss()
classification_criterion = nn.CrossEntropyLoss()

def compute_loss(mask_pred, mask_target, logits, class_target, seg_weight = 1.0, cls_weight = 1.0):
    seg_loss = segmentation_criterion(mask_pred, mask_target)
    cls_loss = classification_criterion(logits, class_target)
    total_loss = seg_weight * seg_loss + cls_weight * cls_loss
    return total_loss, seg_loss, cls_loss

In [14]:
def train_one_epoc(model, loader, optimizer, device):
    model.train()
    running_loss = 0.0

    for images, masks, labels in loader:

        images = images.to(device)
        masks = masks.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        mask_pred, logits = model(images)
        loss, seg_loss, cls_loss = compute_loss(mask_pred, masks, logits, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)

In [15]:
@torch.no_grad()
def validation(model, loader, device):
    model.eval()
    running_loss = 0.0

    all_preds_mask, all_targets_mask = [], []
    all_preds_cls, all_targets_cls = [], []

    for images, masks, labels in loader:
        images = images.to(device)
        masks = masks.to(device)
        labels = labels.to(device)

        mask_preds, logits = model(images)
        loss, _, _, = compute_loss(mask_preds, masks, logits, labels)

        running_loss += loss.item()

        mask_preds_binary = (torch.sigmoid(mask_preds) > 0.5).float()
        cls_pred = torch.argmax(logits, dim=1) # not applying softmax here, since argmax is invariant to monotonic transformations

        all_preds_mask.append(mask_preds_binary.cpu())
        all_targets_mask.append(masks.cpu())
        all_preds_cls.append(cls_pred.cpu())
        all_targets_cls.append(labels.cpu())

    all_preds_mask = torch.cat(all_preds_mask)
    all_targets_mask = torch.cat(all_targets_mask)
    all_preds_cls = torch.cat(all_preds_cls)
    all_targets_cls = torch.cat(all_targets_cls)

    avg_loss = running_loss / len(loader)

    return avg_loss, all_preds_mask, all_targets_mask, all_preds_cls, all_targets_cls

In [16]:
def dice_score(preds, targets, smooth=1e-6):
    intersection = (preds * targets).sum()
    return (2. * intersection + smooth) / (preds.sum() + targets.sum() + smooth) # Dice = (2 * |Prediction ∩ Ground Truth|) / (|Prediction| + |Ground Truth|)

    # smooth is a small constant added to numerator and denominator to prevent division by zero when both masks are empty (common for "notumor" samples), avoiding NaN metrics.

def iou_score(preds, targets, smooth=1e-6):
    intersection = (preds * targets).sum()
    union = preds.sum() + targets.sum() - intersection # Union = prediction area + ground-truth area − intersection
    return (intersection + smooth) / (union + smooth)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def classification_metrics(preds, targets):
    preds_np = preds.numpy()
    targets_np = targets.numpy()
    acc = accuracy_score(targets_np, preds_np)
    precision, recall, f1, _ = precision_recall_fscore_support(
        targets_np, preds_np, average="macro", zero_division=0
    )
    return acc, precision, recall, f1

In [17]:
import time
import torch.optim as optim

print(f"Using device: {device}")

model = UNet(in_channels=1, num_classes=4).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 20
best_val_dice = 0.0
history = {"train_loss": [], "val_loss": [], "val_dice": [], "val_iou": [],
           "val_acc": [], "val_f1": []}

for epoch in range(num_epochs):

    epoch_start = time.time()

    train_loss = train_one_epoc(model, train_loader, optimizer, device)
    val_loss, pm, tm, pc, tc = validation(model, val_loader, device)

    # print(pm.size())
    # print(tm.size())

    dice = dice_score(pm, tm)
    iou = iou_score(pm, tm)
    accuracy, precision, recall, f1 = classification_metrics(pc, tc)

    epoch_end = time.time()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_dice"].append(dice)
    history["val_iou"].append(iou)
    history["val_acc"].append(accuracy)
    history["val_f1"].append(f1)

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | Dice: {dice:.4f} | IoU: {iou:.4f} | "
              f"Val Acc: {accuracy:.4f} | Val F1: {f1:.4f} | "
              f"Epoch Time: {epoch_end - epoch_start:.2f}s")

    if dice > best_val_dice:
        best_val_dice = dice
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  → New best model saved (Dice: {dice:.4f})")

Using device: cuda
Epoch 1/20 | Train Loss: 1.1251 | Val Loss: 0.9666 | Dice: 0.0000 | IoU: 0.0000 | Val Acc: 0.7424 | Val F1: 0.5658 | Epoch Time: 149.23s
  → New best model saved (Dice: 0.0000)
Epoch 2/20 | Train Loss: 0.7623 | Val Loss: 0.6766 | Dice: 0.0000 | IoU: 0.0000 | Val Acc: 0.8299 | Val F1: 0.7628 | Epoch Time: 233.70s
  → New best model saved (Dice: 0.0000)
Epoch 3/20 | Train Loss: 0.6355 | Val Loss: 0.5834 | Dice: 0.0000 | IoU: 0.0000 | Val Acc: 0.8633 | Val F1: 0.7862 | Epoch Time: 185.71s
  → New best model saved (Dice: 0.0000)
Epoch 4/20 | Train Loss: 0.5525 | Val Loss: 0.5865 | Dice: 0.0000 | IoU: 0.0000 | Val Acc: 0.8219 | Val F1: 0.7505 | Epoch Time: 189.55s
Epoch 5/20 | Train Loss: 0.4924 | Val Loss: 0.4592 | Dice: 0.0000 | IoU: 0.0000 | Val Acc: 0.8744 | Val F1: 0.8050 | Epoch Time: 187.58s
  → New best model saved (Dice: 0.0000)
Epoch 6/20 | Train Loss: 0.4527 | Val Loss: 0.4090 | Dice: 0.0002 | IoU: 0.0001 | Val Acc: 0.8712 | Val F1: 0.7970 | Epoch Time: 183.69s

In [18]:
model = UNet(in_channels=1, num_classes=4).to(device)

model.load_state_dict(torch.load("best_model.pth"))
test_loss, pm, tm, pc, tc = validation(model, test_loader, device)  # reusing validate() on test_loader

test_dice = dice_score(pm, tm).item()
test_iou = iou_score(pm, tm).item()
test_acc, test_prec, test_rec, test_f1 = classification_metrics(pc, tc)

print(f"TEST — Dice: {test_dice:.4f} | IoU: {test_iou:.4f} | Acc: {test_acc:.4f} | F1: {test_f1:.4f}")

TEST — Dice: 0.6787 | IoU: 0.5137 | Acc: 0.9253 | F1: 0.8809
